# 2026 COMP90042 Project — Group 073
*Fact-checking: two-stage retrieval (BM25 → BERT cross-encoder) + claim classifier.*

# Readme

**Before running:** upload the project folder to Google Drive at the path set in `PROJECT_ROOT` (cell 1.1).
Required: `data/evidence.json` (~1 GB), `data/train-claims.json`, `data/dev-claims.json`, `data/test-claims-unlabelled.json`.
Optional (saves ~30 min): `cache/bm25_index/`, `cache/bm25_train_top200.json`.

**To resume after a Colab disconnect:** set `RESUME_FROM` in cell 2.1 to the last saved epoch checkpoint path, e.g. `'checkpoints/cross_encoder_epoch1.pt'`.

**Pipeline:** BM25 first-stage retrieval (1.2M → 200 candidates) → BERT cross-encoder reranker (top-4) → claim classifier.

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 1.1 · Setup — Mount Drive, install packages, configure paths
# Run this cell before anything else.

import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

# -- EDIT THIS to match where you uploaded the project on Drive --
PROJECT_ROOT = "/content/drive/MyDrive/COMP90042_2026"
# ----------------------------------------------------------------

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "transformers>=4.40",
        "bm25s[full]>=0.3",
        "scikit-learn>=1.3",
        "tqdm>=4.65",
    ]
)

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# @title 1.2 · Verify data files exist
from pathlib import Path

root = Path(PROJECT_ROOT)
checks = [
    ("data/evidence.json", "required (~1 GB)"),
    ("data/train-claims.json", "required"),
    ("data/dev-claims.json", "required"),
    ("data/test-claims-unlabelled.json", "required"),
    ("cache/bm25_index", "optional — saves ~10 min"),
    ("cache/bm25_train_top200.json", "optional — saves ~20 min"),
]
missing_required = False
for rel, note in checks:
    p = root / rel
    if p.exists():
        size = f"{p.stat().st_size/1e6:.0f} MB" if p.is_file() else "dir"
        print(f"  OK  {rel:45s} {size:10s}  ({note})")
    else:
        icon = "XX" if "required" in note else "--"
        print(f'  {icon}  {rel:45s} {"MISSING":10s}  ({note})')
        if "required" in note:
            missing_required = True

if missing_required:
    raise FileNotFoundError("Required data files are missing. Upload them to Drive first.")

In [ ]:
# @title 1.3 · Build BM25 index — idempotent, skips if cache/bm25_index/ already exists
# Takes ~10 min on first run; subsequent runs print 'already exists' and return.
import importlib.util

spec = importlib.util.spec_from_file_location("build_bm25", f"{PROJECT_ROOT}/scripts/build_bm25.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 2.1 · Train cross-encoder (configure RESUME_FROM / EPOCHS, then run)
#
# After a Colab disconnect:
#   1. Re-run cells 1.1 and 1.2.
#   2. Set RESUME_FROM below to the last saved epoch checkpoint path.
#   3. Re-run this cell.

import importlib.util
import sys

# T4 tip: T4 has 16 GB VRAM. With bert-base-uncased + max_len=256 you can safely
# increase ce_batch_size from 64 to 128 in src/config.py for ~1.5x throughput.

RESUME_FROM = None  # e.g. "checkpoints/cross_encoder_epoch1.pt"  (None = start fresh)
EPOCHS = None  # e.g. 1 to override cfg.ce_epochs=2  (None = use config)

for key in list(sys.modules):
    if key.startswith(("src.", "scripts.")):
        del sys.modules[key]

spec = importlib.util.spec_from_file_location(
    "train_cross_encoder", f"{PROJECT_ROOT}/scripts/train_cross_encoder.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

sys.argv = ["train_cross_encoder.py"]
if RESUME_FROM:
    sys.argv += ["--resume", RESUME_FROM]
if EPOCHS is not None:
    sys.argv += ["--epochs", str(EPOCHS)]

mod.main()
print("\nDone. Final checkpoint: checkpoints/cross_encoder.pt")

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [ ]:
# @title 3.1 · Run inference on dev set (BM25 top-200 -> cross-encoder rerank top-4)
import importlib.util
import sys

SPLIT = "dev"
TOP_K = 4
BM25_TOP_K = 200

for key in list(sys.modules):
    if key.startswith(("src.", "scripts.")):
        del sys.modules[key]

spec = importlib.util.spec_from_file_location(
    "run_inference", f"{PROJECT_ROOT}/scripts/run_inference.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

sys.argv = [
    "run_inference.py",
    "--split",
    SPLIT,
    "--mode",
    "retriever-only",
    "--top-k",
    str(TOP_K),
    "--bm25-top-k",
    str(BM25_TOP_K),
]
mod.main()

In [ ]:
# @title 3.2 · Evaluate on dev set — prints Evidence F, Claim Accuracy, Harmonic Mean
import importlib.util
import sys

PRED_FILE = f"outputs/{SPLIT}-retriever-only-k{TOP_K}.json"
GT_FILE = f"data/{SPLIT}-claims.json"

for key in list(sys.modules):
    if key.startswith(("src.", "scripts.")):
        del sys.modules[key]

spec = importlib.util.spec_from_file_location("eval_script", f"{PROJECT_ROOT}/eval.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

sys.argv = ["eval.py", "--predictions", PRED_FILE, "--groundtruth", GT_FILE, "--verbose"]
mod.main()

In [ ]:
# @title 3.3 · Generate final predictions on test set (for submission)
import importlib.util
import sys

for key in list(sys.modules):
    if key.startswith(("src.", "scripts.")):
        del sys.modules[key]

spec = importlib.util.spec_from_file_location(
    "run_inference", f"{PROJECT_ROOT}/scripts/run_inference.py"
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

sys.argv = [
    "run_inference.py",
    "--split",
    "test",
    "--mode",
    "retriever-only",
    "--top-k",
    "4",
    "--bm25-top-k",
    "200",
]
mod.main()
print("\nTest predictions saved to outputs/test-retriever-only-k4.json")

## Object Oriented Programming codes here

The OOP implementation lives in `src/` (importable modules) and `scripts/` (entry points):

| Module | Key class / function |
|--------|----------------------|
| `src/config.py` | `Config` dataclass — all hyperparameters and paths |
| `src/data_loader.py` | `Claim` dataclass, `load_claims`, `load_evidence` |
| `src/retriever_bm25.py` | `BM25Retriever` — first-stage lexical retrieval |
| `src/retriever_cross_enc.py` | `CrossEncoderHead(nn.Module)` — BERT reranker |
| `src/hard_negatives.py` | `build_training_pairs` — hard-negative mining |
| `src/evaluator.py` | `evaluate` — evidence F, claim accuracy, harmonic mean |
    